In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv('variant2-raisin.csv')

print("Перші 5 рядків датасету:")
display(df.head())

target_col = 'Class' if 'Class' in df.columns else df.columns[-1]

X = df.drop(target_col, axis=1).values
y = df[target_col].values

le = LabelEncoder()
y = le.fit_transform(y)
print(f"\nКласи після кодування: {le.classes_} -> [0, 1]")

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"\nРозміри вибірок:")
print(f"Тренувальна: {X_train.shape[0]} зразків, {X_train.shape[1]} ознак")
print(f"Валідаційна:   {X_val.shape[0]} зразків")
print(f"Тестова:      {X_test.shape[0]} зразків")

Перші 5 рядків датасету:


,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,Extent,Perimeter,Class
0,87524,442.246011,253.291155,0.819738,90546,0.758651,1184.040,Kecimen
1,75166,406.690687,243.032436,0.801805,78789,0.684130,1121.786,Kecimen
2,90856,442.267048,266.328318,0.798354,93717,0.637613,1208.575,Kecimen
3,45928,286.540559,208.760042,0.684989,47336,0.699599,844.162,Kecimen
4,79408,352.190770,290.827533,0.564011,81463,0.792772,1073.251,Kecimen



Класи після кодування: ['Besni' 'Kecimen'] -> [0, 1]

Розміри вибірок:
Тренувальна: 629 зразків, 7 ознак
Валідаційна:   136 зразків
Тестова:      135 зразків


In [8]:
class RaisinDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1) 

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = RaisinDataset(X_train, y_train)
val_dataset = RaisinDataset(X_val, y_val)
test_dataset = RaisinDataset(X_test, y_test)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"DataLoader-и успішно створені!")
print(f"Розмір батчу: {batch_size}")
print(f"Кількість батчів у Train: {len(train_loader)}")
print(f"Кількість батчів у Val:   {len(val_loader)}")

sample_X, sample_y = next(iter(train_loader))
print(f"\nРозмірність батчу X: {sample_X.shape} (32 зразка, 7 ознак)")
print(f"Розмірність батчу y: {sample_y.shape} (32 мітки)")

DataLoader-и успішно створені!
Розмір батчу: 32
Кількість батчів у Train: 20
Кількість батчів у Val:   5

Розмірність батчу X: torch.Size([32, 7]) (32 зразка, 7 ознак)
Розмірність батчу y: torch.Size([32, 1]) (32 мітки)


In [9]:
class DeepClassifier(nn.Module):
    def __init__(self, input_dim):
        super(DeepClassifier, self).__init__()
        
        self.layer1 = nn.Linear(input_dim, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.dropout1 = nn.Dropout(0.3)
        
        self.layer2 = nn.Linear(64, 32)
        self.bn2 = nn.BatchNorm1d(32)
        self.dropout2 = nn.Dropout(0.3)
        
        self.layer3 = nn.Linear(32, 16)
        self.bn3 = nn.BatchNorm1d(16)
        self.dropout3 = nn.Dropout(0.2)
        
        self.layer4 = nn.Linear(16, 8)
        self.bn4 = nn.BatchNorm1d(8)
        self.dropout4 = nn.Dropout(0.2)
        
        self.output_layer = nn.Linear(8, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout1(x)
        
        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout2(x)
        
        x = self.layer3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout3(x)
        
        x = self.layer4(x)
        x = self.bn4(x)
        x = self.relu(x)
        x = self.dropout4(x)
        x = self.output_layer(x)
        return x

input_dim = X_train.shape[1] 

model_demo = DeepClassifier(input_dim)
print("Архітектура нашої глибокої нейромережі успішно створена:")
print(model_demo)

Архітектура нашої глибокої нейромережі успішно створена:
DeepClassifier(
  (layer1): Linear(in_features=7, out_features=64, bias=True)
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (layer2): Linear(in_features=64, out_features=32, bias=True)
  (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (layer3): Linear(in_features=32, out_features=16, bias=True)
  (bn3): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout3): Dropout(p=0.2, inplace=False)
  (layer4): Linear(in_features=16, out_features=8, bias=True)
  (bn4): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout4): Dropout(p=0.2, inplace=False)
  (output_layer): Linear(in_features=8, out_features=1, bias=True)
  (relu): ReLU()
)


In [10]:
import copy

def train_model(model, optimizer, criterion, scheduler, train_loader, val_loader, 
                num_epochs=100, patience=15, l1_lambda=0.0):
    train_losses = []
    val_losses = []
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_weights = copy.deepcopy(model.state_dict())
    
    for epoch in range(num_epochs):
        model.train()
        epoch_train_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)

            if l1_lambda > 0:
                l1_norm = sum(p.abs().sum() for p in model.parameters())
                loss += l1_lambda * l1_norm
                
            loss.backward()
            optimizer.step()
            epoch_train_loss += loss.item() * batch_X.size(0)
            
        avg_train_loss = epoch_train_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)
        
        model.eval()
        epoch_val_loss = 0.0
        
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                epoch_val_loss += loss.item() * batch_X.size(0)
                
        avg_val_loss = epoch_val_loss / len(val_loader.dataset)
        val_losses.append(avg_val_loss)

        scheduler.step(avg_val_loss)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            best_model_weights = copy.deepcopy(model.state_dict()) 
        else:
            epochs_no_improve += 1
            
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Епоха [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | No improve: {epochs_no_improve}/{patience}")
            
        if epochs_no_improve >= patience:
            print(f"\n Early Stopping спрацював на епосі {epoch+1}!")
            print(f"Найкращий Val Loss був: {best_val_loss:.4f}")
            break
            
    print("Завантаження найкращих ваг моделі...")
    model.load_state_dict(best_model_weights)
    
    return train_losses, val_losses, model

print("Функція циклу навчання успішно завантажена!")

Функція циклу навчання успішно завантажена!


In [ ]:
criterion = nn.BCEWithLogitsLoss()

experiments = {
    'Adam (Базовий)':     {'optim': optim.Adam,    'lr': 0.001, 'weight_decay': 0,    'l1': 0},
    'SGD + Momentum':     {'optim': optim.SGD,     'lr': 0.01,  'weight_decay': 0,    'l1': 0, 'momentum': 0.9},
    'RMSprop':            {'optim': optim.RMSprop, 'lr': 0.001, 'weight_decay': 0,    'l1': 0},
    'Adam + L2 (0.01)':   {'optim': optim.Adam,    'lr': 0.001, 'weight_decay': 0.01, 'l1': 0},
    'Adam + L1 (0.001)':  {'optim': optim.Adam,    'lr': 0.001, 'weight_decay': 0,    'l1': 0.001}
}

results = {}
best_overall_name = None
best_overall_val_loss = float('inf')

plt.figure(figsize=(18, 10))

for i, (name, config) in enumerate(experiments.items()):
    print(f"\n{'='*40}")
    print(f" Запуск експерименту: {name}")
    print(f"{'='*40}")
    
    model = DeepClassifier(input_dim)
    if 'momentum' in config:
        optimizer = config['optim'](model.parameters(), lr=config['lr'], 
                                    weight_decay=config['weight_decay'], momentum=config['momentum'])
    else:
        optimizer = config['optim'](model.parameters(), lr=config['lr'], 
                                    weight_decay=config['weight_decay'])
        
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    train_loss, val_loss, best_model = train_model(
        model, optimizer, criterion, scheduler, 
        train_loader, val_loader, 
        num_epochs=150, patience=15, l1_lambda=config['l1']
    )
    
    results[name] = best_model
    
    current_best_val_loss = min(val_loss)
    if current_best_val_loss < best_overall_val_loss:
        best_overall_val_loss = current_best_val_loss
        best_overall_name = name

    plt.subplot(2, 3, i+1)
    plt.plot(train_loss, label='Train Loss', color='blue')
    plt.plot(val_loss, label='Val Loss', color='red')
    plt.title(name)
    plt.xlabel('Епохи')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

print("\n" + "*"*50)
print("ОЦІНКА НАЙКРАЩОЇ МОДЕЛІ (МЕТРИКИ ТА ROC-КРИВА)")
print("*"*50)

print(f"🏆 Динамічно обрана найкраща модель: {best_overall_name} (Best Val Loss: {best_overall_val_loss:.4f})")

best_final_model = results[best_overall_name]
best_final_model.eval()

y_true = []
y_scores = []
y_pred = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = best_final_model(batch_X)
        probs = torch.sigmoid(outputs) 
        
        preds = (probs > 0.5).float()
        
        y_true.extend(batch_y.numpy())
        y_scores.extend(probs.numpy())
        y_pred.extend(preds.numpy())

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy (Точність): {acc:.4f}")
print(f"Precision (Влучність): {prec:.4f}")
print(f"Recall (Повнота): {rec:.4f}")
print(f"F1-score (Гармонійне середнє): {f1:.4f}")

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (Хибно позитивні)')
plt.ylabel('True Positive Rate (Істинно позитивні)')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


 Запуск експерименту: Adam (Базовий)
Епоха [1/150] | Train Loss: 0.6496 | Val Loss: 0.6480 | No improve: 0/15
Епоха [10/150] | Train Loss: 0.4542 | Val Loss: 0.4643 | No improve: 0/15
Епоха [20/150] | Train Loss: 0.3803 | Val Loss: 0.4153 | No improve: 1/15
Епоха [30/150] | Train Loss: 0.3832 | Val Loss: 0.3903 | No improve: 0/15
Епоха [40/150] | Train Loss: 0.3628 | Val Loss: 0.3921 | No improve: 8/15

 Early Stopping спрацював на епосі 47!
Найкращий Val Loss був: 0.3818
Завантаження найкращих ваг моделі...

 Запуск експерименту: SGD + Momentum
Епоха [1/150] | Train Loss: 0.6832 | Val Loss: 0.6048 | No improve: 0/15
Епоха [10/150] | Train Loss: 0.3829 | Val Loss: 0.3974 | No improve: 0/15
Епоха [20/150] | Train Loss: 0.3867 | Val Loss: 0.3890 | No improve: 2/15
Епоха [30/150] | Train Loss: 0.3462 | Val Loss: 0.3826 | No improve: 12/15
Епоха [40/150] | Train Loss: 0.3752 | Val Loss: 0.3798 | No improve: 8/15

 Early Stopping спрацював на епосі 47!
Найкращий Val Loss був: 0.3788
Завант

KeyboardInterrupt: 